# Hybrid Search - Langchain


In [21]:
%pip install --upgrade --quiet pinecone pinecone-client pinecone-text pinecone-notebooks


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [34]:
import pinecone
print(pinecone.__version__)

9.0.0


In [35]:
%pip uninstall pinecone -y
%pip install pinecone==3.2.2

Note: you may need to restart the kernel to use updated packages.
  Attempting uninstall: pinecone-client
    Found existing installation: pinecone-client 6.0.0
    Uninstalling pinecone-client-6.0.0:
      Successfully uninstalled pinecone-client-6.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pinecone]

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.environ['PINECONE_API_KEY']

In [38]:
from langchain_community.retrievers import PineconeHybridSearchRetriever # It has both semantic & syntactic search
from pinecone import Pinecone, ServerlessSpec
index_name = "hyd-search-langchain-pinecone"

## initialize the pincone client
pc = Pinecone(api_key=api_key)

# create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension= 384, 
        metric= 'dotproduct', #sparse values supported only for dotproduct
        spec= ServerlessSpec(cloud='aws', region='us-east-1')
    )



In [39]:
index = pc.Index(index_name)
index

Index(host='https://hyd-search-langchain-pinecone-t90veoi.svc.aped-4627-b74a.pinecone.io')

In [40]:
## vector embedding and sparse matrix
%pip install langchain-huggingface sentence-transformers nltk


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10312.06it/s]


In [42]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nithingedda/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [43]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [44]:
sentences = [
    "In 2024, I visited Paris",
    "In 2025, I visited Italy",
    "In 2026, I visited Kenya",
    "In 2027, I visited India"
]

# tfidf values on the sentences
bm25_encoder.fit(sentences)

# store values to json file
bm25_encoder.dump("bm25_values.json")

# load to ur BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")


100%|██████████| 4/4 [00:00<00:00, 17172.18it/s]


In [45]:
retriever = PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(), sparse_encoder=bm25_encoder,index=index)
retriever

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8775.43it/s]


PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x12c8e6fd0>, index=Index(host='https://hyd-search-langchain-pinecone-t90veoi.svc.aped-4627-b74a.pinecone.io'))

In [ ]:
retriever.add_texts(
    [ "In 2024, I visited Paris",
    "In 2025, I visited Italy",
    "In 2026, I visited Kenya",
    "In 2027, I visited India"
    ]
)

In [ ]:
retriever.invoke("Which country did I visit in 2025?")
